# Intialisation step

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

In [0]:
RENAME_MAP = {
    "cst_id": "CustomerID",
    "cst_key": "key",
    "cst_firstname": "FirstName",
    "cst_lastname": "LastName",
    "cst_marital_status": "MaritalStatus",
    "cst_gndr": "Gender",
    "cst_create_date": "CreateDate",
}

# Reading from the Bronze

In [0]:
df =   spark.table("workspace.bronze.crm_cust_info")

In [0]:
df.display()

# Data Transformation

## Trimming

In [0]:


# Trim the string (generic — applies to every string column, not just names)
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


In [0]:
df.display()

## Normalisation

In [0]:

df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

In [0]:
df.display()

## Renaming the column 

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)



In [0]:
df.display()

# Write into silver

In [0]:
(
    df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.crm_customer")
)